In [9]:
import pandas as pd
import numpy as np

# 模拟从 ODS 层（原始数据源）导出的未加任何处理的脏日志
data = {
    'user_id': [' u-1029 ', 'U-8812', 'u_9921', 'U-1029', 'sys_test_01'], # 包含空格、大小写不一、分隔符不一、重复用户、测试账号
    'signup_date': ['2023-11-01 14:30:00', '15/12/2023', '2024.01.20', '2023-11-01 14:30:00', '1970-01-01'], # 格式混乱、带有具体时分秒、重复时间、系统幽灵时间
    'monthly_fee': ['$19.99', '19.99', '¥140.00', '19.9', '-9.99'], # 币种混杂、纯数字、退款/黑产数据
    'account_status': ['Active', 'CANCELED', ' pending ', 'Active', 'Admin'], # 大小写不统一、带有隐藏空格、非正常的业务状态
    'user_age': ['28', '35.5', '999', '28', 'NaN'] # 字符串型的数字、带有小数的年龄、老妖精占位符、字符串型的缺失值
}
df_saas_raw = pd.DataFrame(data)
print("📥 原始数据已拉取完毕：")
print(df_saas_raw)

📥 原始数据已拉取完毕：
       user_id          signup_date monthly_fee account_status user_age
0      u-1029   2023-11-01 14:30:00      $19.99         Active       28
1       U-8812           15/12/2023       19.99       CANCELED     35.5
2       u_9921           2024.01.20     ¥140.00       pending       999
3       U-1029  2023-11-01 14:30:00        19.9         Active       28
4  sys_test_01           1970-01-01       -9.99          Admin      NaN


In [ ]:
# 备份数据
df_clean = df_saas_raw.copy()

# 提取user_id
df_clean['user_id'] = df_clean['user_id'].str.replace(r'\D+','',regex=True)
print(df_clean['user_id'])

# 处理日期

# 抽样查看
date_type = df_clean['signup_date'].value_counts()
print(date_type)

# 第一步：主力格式解析
df_clean['parsed_date'] = pd.to_datetime(df_clean['signup_date'], errors='coerce', format='%Y-%m-%d')

# 第二步：抓取斜杠格式
mask = df_clean['parsed_date'].isna() # 第一次生成掩码，锁定剩下的 4 个
df_clean.loc[mask, 'parsed_date'] = pd.to_datetime(df_clean.loc[mask, 'signup_date'], errors='coerce', format='%Y/%m/%d')

# 第三步：抓取欧洲格式
mask = df_clean['parsed_date'].isna() # 【关键！】刷新掩码，此时第 0 行已经被保护起来了，只锁定剩下的 3 个
df_clean.loc[mask, 'parsed_date'] = pd.to_datetime(df_clean.loc[mask, 'signup_date'], errors='coerce', format='%d-%m-%Y')

# 第四步：抓取美国格式
mask = df_clean['parsed_date'].isna() # 【关键！】再次刷新掩码，只锁定剩下的 2 个
df_clean.loc[mask, 'parsed_date'] = pd.to_datetime(df_clean.loc[mask, 'signup_date'], errors='coerce', format='%m/%d/%Y')

print(df_clean[['signup_date', 'parsed_date']])

# 清洗monthly_fee

# 第一步：创建临时解析变量
temp_parsed = pd.to_numeric(df_clean['monthly_fee'],errors='coerce')

# 第二步：创建NaN布尔序列
dirty_mask = temp_parsed.isna()

# 第三步：找出被解析为空值的原数据
raw_dirty_data = df_clean.loc[dirty_mask,'monthly_fee']
print(f"原始脏数据：")
print(raw_dirty_data.unique())

# 第四步：正则提取被转换为空值的原始脏数据中的有效数值
df_clean['monthly_fee'] = df_clean.loc['monthly_fee'].astype(str).str.replace(r'[^\d\.\-]','',regex=True)


0    1029
1    8812
2    9921
3    1029
4      01
Name: user_id, dtype: object
signup_date
2023-11-01 14:30:00    2
15/12/2023             1
2024.01.20             1
1970-01-01             1
Name: count, dtype: int64
           signup_date parsed_date
0  2023-11-01 14:30:00         NaT
1           15/12/2023         NaT
2           2024.01.20         NaT
3  2023-11-01 14:30:00         NaT
4           1970-01-01  1970-01-01


KeyError: 'total_spend'